# RIKUY_DEEP: deteccion de defectos en piezas de fundicion

Flujo: ingesta y versionado del dataset (data-v1) -> CNN (cnn-v1) -> ResNet50 (resnet-v1) -> comparacion en test -> exportacion a TensorFlow.js.

Decisiones de diseno:

El dataset de Kaggle ya viene augmentado: no se aplica augmentation durante el entrenamiento.
data-v1 es el dataset tal como se descarga. El redimensionado ocurre en tf.data, no se duplica en disco.
Clase positiva (1) = defecto. El modelo devuelve P(defecto).
Validacion se usa para early stopping y umbral; test solo para el reporte final.

In [37]:
%pip install -q kagglehub dvc dvc-gdrive tensorflowjs

## LIBRERIAS

In [38]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from datetime import datetime, timezone
from pathlib import Path
from PIL import Image
from google.colab import drive
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report,confusion_matrix, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator , load_img ,img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense, Conv2D, MaxPooling2D

## CONFIGURACIÓN KAGGLE Y GOOGLE DRIVE

In [39]:
drive.mount('/content/drive')

DATA_VERSION, CNN_VERSION, RESNET_VERSION = 'data-v1', 'cnn-v1', 'resnet-v1'
KAGGLE_DATASET = 'ravirajsinh45/real-life-industrial-dataset-of-casting-product'
IMAGE_EXTENSIONS = {'.jpeg', '.jpg', '.png'}

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'RIKUY_DEEP'
REPO_DIR = DRIVE_ROOT / 'ColabNotebooks' / 'RIKUY_DEEP_G2'
DATA_DIR = PROJECT_DIR / 'data'
RAW_DATA = DATA_DIR / 'casting_data'
REPO_DATA_DIR = REPO_DIR / 'data'
REMOTE_DIR = REPO_DIR / 'dvc_remote'
REGISTRY_DIR = PROJECT_DIR / 'model_registry'
EXPORT_DIR = PROJECT_DIR / 'exports'
SPLITS_CSV = PROJECT_DIR / 'splits.csv'

for directory in (PROJECT_DIR, DATA_DIR, RAW_DATA, REPO_DIR, REGISTRY_DIR, EXPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)
print(f'Proyecto y fuente de datos: {PROJECT_DIR}')
print(f'Datos crudos: {RAW_DATA}')
print(f'Repositorio: {REPO_DIR}')
print(f'Datos DVC: {REPO_DATA_DIR}')
print(f'Remoto DVC en Google Drive: {REMOTE_DIR}')
print(f'Registro de modelos: {REGISTRY_DIR}')
print(f'Modelos exportados: {EXPORT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto y fuente de datos: /content/drive/MyDrive/RIKUY_DEEP
Datos crudos: /content/drive/MyDrive/RIKUY_DEEP/data/casting_data
Repositorio: /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2
Datos DVC: /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2/data
Remoto DVC en Google Drive: /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2/dvc_remote
Registro de modelos: /content/drive/MyDrive/RIKUY_DEEP/model_registry
Modelos exportados: /content/drive/MyDrive/RIKUY_DEEP/exports


## Descargando el dataset

In [40]:
if not (RAW_DATA / 'def_front').exists() or not (RAW_DATA / 'ok_front').exists():
    print("Descargando dataset desde Kaggle con kagglehub...")
    path = Path(kagglehub.dataset_download(KAGGLE_DATASET))

    # Localizar la carpeta que contiene directamente las clases
    candidates = [
        path / 'casting_512x512' / 'casting_512x512',
        path / 'casting_512x512',
        path
    ]
    source_dir = next((c for c in candidates if (c / 'def_front').exists() and (c / 'ok_front').exists()), None)

    if source_dir is None:
        raise FileNotFoundError(f"No se encontró la carpeta 'casting_512x512' con las clases en {path}")

    print(f"Copiando imágenes 512x512 desde {source_dir} hacia {RAW_DATA}...")
    shutil.copytree(source_dir, RAW_DATA, dirs_exist_ok=True)
    print("Copia finalizada.")
else:
    print(f"ℹEl dataset 512x512 ya existe en {RAW_DATA}")
# Verificación y conteo de imágenes (Criterio de aceptación HU04)
num_def = len(list((RAW_DATA / 'def_front').glob('*.*')))
num_ok = len(list((RAW_DATA / 'ok_front').glob('*.*')))
total = num_def + num_ok

ℹEl dataset 512x512 ya existe en /content/drive/MyDrive/RIKUY_DEEP/data/casting_data


## Versionando el dataset

In [41]:
REPO_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
legacy_data_dir = REPO_DATA_DIR / 'data' / 'casting_data'
tracked_data_dir = REPO_DATA_DIR / 'casting_data'
REPO_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Repair the previous sync layout without discarding the existing dataset.
if legacy_data_dir.is_dir() and not tracked_data_dir.exists():
    shutil.move(str(legacy_data_dir), str(tracked_data_dir))

subprocess.run([
    'rsync', '-ah', '--progress', str(RAW_DATA) + '/', str(tracked_data_dir) + '/'
], check=True)
print(f'Datos preparados para DVC en: {tracked_data_dir}')

In [ ]:
%cd {REPO_DIR}

In [ ]:
gitignore_path = REPO_DIR / '.gitignore'
gitignore_text = gitignore_path.read_text()
if '/dvc_remote/' not in gitignore_text.splitlines():
    gitignore_path.write_text(gitignore_text.rstrip('\n') + '\n/dvc_remote/\n')
subprocess.run(['git', 'add', '.gitignore'], cwd=REPO_DIR, check=True)
status = subprocess.run(['git', 'status', '--short'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
print(status.stdout)

In [ ]:
REMOTE_DIR.mkdir(parents=True, exist_ok=True)
remote_list = subprocess.run(
    ['dvc', 'remote', 'list'], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.splitlines()
if any(line.split() and line.split()[0] == 'drive' for line in remote_list):
    subprocess.run(['dvc', 'remote', 'modify', 'drive', 'url', str(REMOTE_DIR)], cwd=REPO_DIR, check=True)
    subprocess.run(['dvc', 'remote', 'default', 'drive'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['dvc', 'remote', 'add', '--default', 'drive', str(REMOTE_DIR)], cwd=REPO_DIR, check=True)

subprocess.run(['dvc', 'add', 'data'], cwd=REPO_DIR, check=True)
subprocess.run(['dvc', 'push'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'add', 'data.dvc', '.gitignore', '.dvc/config'], cwd=REPO_DIR, check=True)
status = subprocess.run(
    ['git', 'status', '--short'], cwd=REPO_DIR, check=True, capture_output=True, text=True
)
print(status.stdout)

## Preprocesamiento de imagenes | Data augmentation

In [ ]:
%cd /content/drive/MyDrive/ColabNotebooks/RIKUY_DEEP_G2

In [ ]:
!git fetch origin


In [ ]:
!git switch fix-HU04

In [ ]:
!git status

In [ ]:
# 1. Añadir los cambios al staging
!git add RIKUY_DEEP_G2.ipynb

# 2. Crear el commit siguiendo Conventional Commits (tipo fix)!
!git commit -m "(fix): control de versiones con DVC"

# 3. Realizar push a la rama fix-HU04
!git push origin fix-HU04